In [ ]:
#5a. Connecting Tkinter with SQL code for writing data into tables and reading contents of tables.

import tkinter as tk
from tkinter import ttk
from tkinter import messagebox
import sqlite3

# --------------------------------------------
# DATABASE CONNECTION
# --------------------------------------------

db_file = "example_ship.db"
conn = sqlite3.connect(db_file)
cursor = conn.cursor()

table_name = "passengers"
create_table_sql = """
    CREATE TABLE IF NOT EXISTS passengers (
        passenger_id INTEGER PRIMARY KEY,
        first_name TEXT NOT NULL,
        last_name TEXT NOT NULL,
        age INTEGER,
        nationality TEXT
    );"""

#helper functions
def connect_to_database(db_file):
    con = sqlite3.connect(db_file)
    print (con)
    return con
    
db_base = connect_to_database(db_file)
cursor = db_base.cursor()
cursor.execute(create_table_sql)
conn.commit()

# -------------------------------------------------
# FUNCTION TO READ DATA AND DISPLAY IN TREEVIEW
# -------------------------------------------------

def read_data_from_database_to_form(cursor, table_name):
    """
    Executes a SELECT query on the specified table
    and displays the results in the Tkinter Treeview.
    """
    try:
        # Execute query
        cursor.execute(f"SELECT * FROM  {table_name}")   
        
        # Fetch all rows returned by the query
        rows = cursor.fetchall()
        
        # Get column names from the query result
        columns = [description[0] for description in cursor.description]

        # Clear any existing rows from the Treeview
        for item in tree.get_children():
            tree.delete(item)
        
        # Configure Treeview columns
        tree["columns"] = columns
        tree["show"] = "headings"

        # Create column headings dynamically
        for col in columns:
            tree.heading(col, text=col)
            tree.column(col, width=120)

        # Insert each row into the Treeview
        for row in rows:
            tree.insert("", tk.END, values=row)

    except Exception as e:
        # Print any errors that occur
        print("Error:", e)
    
    # Print results for debugging
    for row in rows:
        print(row)
        
# --------------------------------------------
# FUNCTION TO INSERT DATA
# --------------------------------------------

def insert_data():
    try:
        # Get values from entry fields
        first_name = entry_first_name.get()
        last_name = entry_last_name.get()
        age = entry_age.get()
        nationality = entry_nationality.get()
        #ticket_number = entry_ticket.get()
        #cabin = entry_cabin.get()

        # Basic validation
        if not first_name or not last_name:
            messagebox.showerror("Error", "First and Last name are required.")
            return
        if not(age.isdigit()):
            messagebox.showerror("Error", "Age must be an integer value")
            return
            
        # Insert into database
        cursor.execute("""
            INSERT INTO passengers 
            (first_name, last_name, age, nationality)
            VALUES (?, ?, ?, ?)
        """, (first_name, last_name, age, nationality))

        conn.commit()

        messagebox.showinfo("Success", "Record inserted successfully.")

        # Clear fields after insert
        entry_first_name.delete(0, tk.END)
        entry_last_name.delete(0, tk.END)
        entry_age.delete(0, tk.END)
        entry_nationality.delete(0, tk.END)
        #entry_ticket.delete(0, tk.END)
        #entry_cabin.delete(0, tk.END)

    except Exception as e:
        messagebox.showerror("Database Error", str(e))


# --------------------------------------------
# GUI SETUP
# --------------------------------------------

root = tk.Tk()
root.title("Passenger Entry Form")
root.geometry("400x350")

# creating new frame for passenger form
ps_frame = tk.Frame(root, width = 300, height = 300, bg = 'white')
ps_frame.pack(padx = 10, pady = 10)

# Labels and Entry Fields
tk.Label(ps_frame, text="First Name").pack()
entry_first_name = tk.Entry(ps_frame)
entry_first_name.pack()

tk.Label(ps_frame, text="Last Name").pack()
entry_last_name = tk.Entry(ps_frame)
entry_last_name.pack()

tk.Label(ps_frame, text="Age").pack()
entry_age = tk.Entry(ps_frame)
entry_age.pack()

tk.Label(ps_frame, text="Nationality").pack()
entry_nationality = tk.Entry(ps_frame)
entry_nationality.pack()

# Submit Button
tk.Button(ps_frame, text="Insert Record", command=insert_data).pack(pady=10)

# Button to load data
load_button = tk.Button(root, text="Load Data", command=lambda: read_data_from_database_to_form(cursor, table_name))
load_button.pack(pady=10)

# Frame for table and scrollbar
frame = tk.Frame(root, bg = 'lightblue')
frame.pack(fill="both", expand=True)

# Create vertical and horizontal scrollbars
scroll_y = tk.Scrollbar(frame, orient="vertical")
scroll_x = tk.Scrollbar(frame, orient="horizontal")

# Create Treeview widget (table display)
tree = ttk.Treeview(
    frame,
    yscrollcommand=scroll_y.set,
    xscrollcommand=scroll_x.set
)

# Connect scrollbars to Treeview
scroll_y.config(command=tree.yview)
scroll_x.config(command=tree.xview)

# Pack scrollbars and table into frame
scroll_y.pack(side="right", fill="y")
scroll_x.pack(side="bottom", fill="x")
tree.pack(fill="both", expand=True)

# Start GUI
root.configure(bg = 'lightblue')
root.mainloop()

# Close DB connection after window closes
conn.close()